# Magnetic Hyperthermia Simulation

This notebook simulates magnetic hyperthermia treatment using physical equations.

## Theory

Magnetic hyperthermia is a cancer treatment technique where magnetic nanoparticles are injected into tumor tissue and heated using an alternating magnetic field (AMF). The heat generation depends on:

### 1. Specific Absorption Rate (SAR)

The SAR represents the power absorbed per unit mass of magnetic material:

$$SAR = \frac{P}{m} = \pi \mu_0 \chi'' H^2 f$$

where:
- $\mu_0$ = permeability of free space (4π × 10⁻⁷ H/m)
- $\chi''$ = imaginary part of magnetic susceptibility
- $H$ = magnetic field amplitude (A/m)
- $f$ = frequency of alternating magnetic field (Hz)

### 2. Linear Response Theory (LRT)

For superparamagnetic nanoparticles in the linear response regime:

$$SAR = \pi \mu_0 \chi_0 H^2 f \frac{2\pi f \tau}{1 + (2\pi f \tau)^2}$$

where:
- $\chi_0$ = equilibrium susceptibility
- $\tau$ = relaxation time (Néel or Brownian)

### 3. Temperature Evolution

The temperature change in tissue is described by the bio-heat equation (simplified):

$$\rho C_p \frac{dT}{dt} = SAR \cdot c_{NP} - k(T - T_0)$$

where:
- $\rho$ = tissue density (kg/m³)
- $C_p$ = specific heat capacity (J/kg·K)
- $c_{NP}$ = nanoparticle concentration (kg/m³)
- $k$ = heat loss coefficient
- $T_0$ = body temperature (37°C)

## Install and Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.constants import pi, mu_0

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Define Physical Parameters

In [ ]:
# Magnetic nanoparticle parameters
d = 15e-9  # Particle diameter (m) - 15 nm
Ms = 4.46e5  # Saturation magnetization (A/m) for magnetite
K = 1.35e4  # Anisotropy constant (J/m³) for magnetite
V = (pi * d**3) / 6  # Particle volume (m³)

# Magnetic field parameters
H = 1.5e4  # Magnetic field amplitude (A/m) - ~19 kA/m
f = 3e5  # Frequency (Hz) - 300 kHz

# Relaxation parameters
tau_0 = 1e-9  # Attempt time (s)
k_B = 1.38e-23  # Boltzmann constant (J/K)
T_ref = 310  # Reference temperature (K) - 37°C

# Calculate Néel relaxation time
E_barrier = K * V  # Energy barrier
tau_N = tau_0 * np.exp(E_barrier / (k_B * T_ref))  # Néel relaxation time

# Calculate equilibrium susceptibility (Langevin theory)
chi_0 = (mu_0 * Ms**2 * V) / (3 * k_B * T_ref)  # Dimensionless

# Tissue parameters
rho_tissue = 1050  # Tissue density (kg/m³)
Cp_tissue = 3600  # Specific heat capacity (J/kg·K)
c_NP = 5.0  # Nanoparticle concentration (kg_NP/m³_tissue)
rho_NP = 5200  # Nanoparticle density (kg/m³) - magnetite

# Heat loss parameters
k_loss = 5.0  # Heat loss coefficient (W/m³·K)
T_body = 310.15  # Body temperature (K) - 37°C

# Simulation parameters
t_max = 1800  # Maximum simulation time (s) - 30 minutes
dt = 1  # Time step (s)

print(f"Particle diameter: {d*1e9:.1f} nm")
print(f"Particle volume: {V*1e27:.2f} nm³")
print(f"Energy barrier: {E_barrier/k_B:.1f} K")
print(f"Néel relaxation time: {tau_N:.2e} s")
print(f"Equilibrium susceptibility: {chi_0:.2e}")
print(f"Applied field: {H*1e-3:.1f} kA/m")
print(f"Frequency: {f*1e-3:.0f} kHz")
print(f"NP concentration: {c_NP:.1f} kg/m³")

## Calculate Specific Absorption Rate (SAR)

In [ ]:
def calculate_SAR_LRT(H, f, chi_0, tau):
    """
    Calculate SAR using Linear Response Theory
    
    Parameters:
    -----------
    H : float
        Magnetic field amplitude (A/m)
    f : float
        Frequency (Hz)
    chi_0 : float
        Equilibrium susceptibility (dimensionless)
    tau : float
        Relaxation time (s)
    
    Returns:
    --------
    SAR : float
        Specific Absorption Rate (W/kg)
    """
    omega = 2 * pi * f
    chi_double_prime = chi_0 * (omega * tau) / (1 + (omega * tau)**2)
    
    # Power dissipation per unit volume
    P_vol = pi * mu_0 * chi_double_prime * H**2 * f
    
    # Convert to SAR (W/kg)
    SAR = P_vol / rho_NP
    
    return SAR

# Calculate SAR
SAR = calculate_SAR_LRT(H, f, chi_0, tau_N)

print(f"\nSpecific Absorption Rate (SAR):")
print(f"SAR = {SAR:.2f} W/kg")
print(f"\nThis is equivalent to:")
print(f"- {SAR*1e-3:.2f} kW/kg")
print(f"- {SAR*rho_NP*1e-6:.2f} MW/m³ (volumetric power)")

## SAR vs Frequency Analysis

In [ ]:
# Sweep frequency from 10 kHz to 1 MHz
frequencies = np.logspace(4, 6, 100)  # 10 kHz to 1 MHz
SAR_values = [calculate_SAR_LRT(H, freq, chi_0, tau_N) for freq in frequencies]

plt.figure(figsize=(10, 6))
plt.semilogx(frequencies*1e-3, SAR_values, 'b-', linewidth=2)
plt.axvline(f*1e-3, color='r', linestyle='--', label=f'Operating frequency: {f*1e-3:.0f} kHz')
plt.xlabel('Frequency (kHz)', fontsize=12)
plt.ylabel('SAR (W/kg)', fontsize=12)
plt.title('Specific Absorption Rate vs Frequency', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Find optimal frequency
max_idx = np.argmax(SAR_values)
print(f"Optimal frequency for maximum SAR: {frequencies[max_idx]*1e-3:.1f} kHz")
print(f"Maximum SAR: {SAR_values[max_idx]:.2f} W/kg")

## SAR vs Magnetic Field Analysis

In [ ]:
# Sweep magnetic field from 1 to 30 kA/m
H_values = np.linspace(1e3, 3e4, 100)  # 1 to 30 kA/m
SAR_H = [calculate_SAR_LRT(h, f, chi_0, tau_N) for h in H_values]

plt.figure(figsize=(10, 6))
plt.plot(H_values*1e-3, SAR_H, 'g-', linewidth=2)
plt.axvline(H*1e-3, color='r', linestyle='--', label=f'Operating field: {H*1e-3:.1f} kA/m')
plt.xlabel('Magnetic Field Amplitude (kA/m)', fontsize=12)
plt.ylabel('SAR (W/kg)', fontsize=12)
plt.title('Specific Absorption Rate vs Magnetic Field', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Temperature Evolution Simulation

In [ ]:
def temperature_evolution(T, t, SAR, c_NP, rho_tissue, Cp_tissue, k_loss, T_body):
    """
    Bio-heat equation for temperature evolution
    
    Parameters:
    -----------
    T : float
        Temperature (K)
    t : float
        Time (s)
    SAR : float
        Specific Absorption Rate (W/kg)
    c_NP : float
        Nanoparticle concentration (kg/m³)
    rho_tissue : float
        Tissue density (kg/m³)
    Cp_tissue : float
        Tissue specific heat (J/kg·K)
    k_loss : float
        Heat loss coefficient (W/m³·K)
    T_body : float
        Body temperature (K)
    
    Returns:
    --------
    dT/dt : float
        Temperature rate of change (K/s)
    """
    heat_generation = SAR * c_NP
    heat_loss = k_loss * (T - T_body) / (rho_tissue * Cp_tissue)
    dTdt = (heat_generation / (rho_tissue * Cp_tissue)) - heat_loss
    return dTdt

# Time array
time = np.arange(0, t_max, dt)

# Solve ODE
T_initial = T_body  # Start at body temperature
temperature = odeint(temperature_evolution, T_initial, time, 
                     args=(SAR, c_NP, rho_tissue, Cp_tissue, k_loss, T_body))

# Convert to Celsius
T_celsius = temperature.flatten() - 273.15
time_minutes = time / 60

print(f"Initial temperature: {T_celsius[0]:.2f}°C")
print(f"Final temperature (after {t_max/60:.0f} min): {T_celsius[-1]:.2f}°C")
print(f"Temperature increase: {T_celsius[-1] - T_celsius[0]:.2f}°C")
print(f"\nTherapeutic range: 41-46°C")
if T_celsius[-1] >= 41 and T_celsius[-1] <= 46:
    print("✓ Temperature is within therapeutic range!")
elif T_celsius[-1] < 41:
    print("⚠ Temperature is below therapeutic range")
else:
    print("⚠ Temperature exceeds therapeutic range")

## Plot Temperature Evolution

In [ ]:
plt.figure(figsize=(12, 7))

# Plot temperature evolution
plt.plot(time_minutes, T_celsius, 'b-', linewidth=2.5, label='Tumor temperature')

# Mark therapeutic window
plt.axhspan(41, 46, alpha=0.2, color='green', label='Therapeutic range (41-46°C)')
plt.axhline(37, color='gray', linestyle='--', alpha=0.5, label='Body temperature (37°C)')
plt.axhline(43, color='orange', linestyle='--', alpha=0.7, label='Target temperature (43°C)')

plt.xlabel('Time (minutes)', fontsize=13)
plt.ylabel('Temperature (°C)', fontsize=13)
plt.title('Temperature Evolution during Magnetic Hyperthermia Treatment', 
          fontsize=15, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11, loc='best')
plt.xlim(0, t_max/60)
plt.tight_layout()
plt.show()

## Parametric Study: Effect of Nanoparticle Concentration

In [ ]:
# Test different nanoparticle concentrations
concentrations = [1.0, 2.5, 5.0, 7.5, 10.0]  # kg/m³
colors = ['blue', 'green', 'orange', 'red', 'purple']

plt.figure(figsize=(12, 7))

for c, color in zip(concentrations, colors):
    temp = odeint(temperature_evolution, T_initial, time, 
                  args=(SAR, c, rho_tissue, Cp_tissue, k_loss, T_body))
    temp_c = temp.flatten() - 273.15
    plt.plot(time_minutes, temp_c, color=color, linewidth=2, 
             label=f'c = {c:.1f} kg/m³')

# Mark therapeutic window
plt.axhspan(41, 46, alpha=0.15, color='green', label='Therapeutic range')
plt.axhline(37, color='gray', linestyle='--', alpha=0.5)

plt.xlabel('Time (minutes)', fontsize=13)
plt.ylabel('Temperature (°C)', fontsize=13)
plt.title('Effect of Nanoparticle Concentration on Temperature', 
          fontsize=15, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10, loc='best')
plt.xlim(0, t_max/60)
plt.tight_layout()
plt.show()

## Parametric Study: Effect of Particle Size

In [ ]:
# Test different particle sizes
particle_sizes = np.array([10, 12, 15, 18, 20]) * 1e-9  # nm to m
colors_size = ['purple', 'blue', 'green', 'orange', 'red']

plt.figure(figsize=(12, 7))

SAR_values_size = []

for d_p, color in zip(particle_sizes, colors_size):
    V_p = (pi * d_p**3) / 6
    E_barrier_p = K * V_p
    tau_N_p = tau_0 * np.exp(E_barrier_p / (k_B * T_ref))
    chi_0_p = (mu_0 * Ms**2 * V_p) / (3 * k_B * T_ref)
    
    SAR_p = calculate_SAR_LRT(H, f, chi_0_p, tau_N_p)
    SAR_values_size.append(SAR_p)
    
    temp = odeint(temperature_evolution, T_initial, time, 
                  args=(SAR_p, c_NP, rho_tissue, Cp_tissue, k_loss, T_body))
    temp_c = temp.flatten() - 273.15
    plt.plot(time_minutes, temp_c, color=color, linewidth=2, 
             label=f'd = {d_p*1e9:.0f} nm (SAR = {SAR_p:.1f} W/kg)')

# Mark therapeutic window
plt.axhspan(41, 46, alpha=0.15, color='green', label='Therapeutic range')
plt.axhline(37, color='gray', linestyle='--', alpha=0.5)

plt.xlabel('Time (minutes)', fontsize=13)
plt.ylabel('Temperature (°C)', fontsize=13)
plt.title('Effect of Particle Size on Temperature Evolution', 
          fontsize=15, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10, loc='best')
plt.xlim(0, t_max/60)
plt.tight_layout()
plt.show()

print("\nSAR values for different particle sizes:")
for d_p, sar in zip(particle_sizes, SAR_values_size):
    print(f"d = {d_p*1e9:.0f} nm: SAR = {sar:.2f} W/kg")

## Summary and Clinical Considerations

### Key Results:
- The simulation models heat generation from magnetic nanoparticles under alternating magnetic fields
- Temperature evolution follows the bio-heat equation with heat generation and loss terms
- Therapeutic temperature range for hyperthermia: **41-46°C**
- Target treatment temperature: **43°C**

### Adjustable Parameters:
1. **Magnetic Field Parameters**: Amplitude (H) and frequency (f)
2. **Nanoparticle Properties**: Size, material (Ms, K), concentration
3. **Treatment Duration**: Simulation time
4. **Tissue Properties**: Density, specific heat, perfusion rate

### Clinical Safety:
- Maximum field-frequency product: H·f < 5×10⁹ A·m⁻¹·s⁻¹ (Brezovich criterion)
- Treatment duration: typically 30-60 minutes
- Temperature monitoring is critical during treatment

### References:
1. Rosensweig, R. E. (2002). "Heating magnetic fluid with alternating magnetic field." Journal of Magnetism and Magnetic Materials.
2. Hergt, R., & Dutz, S. (2007). "Magnetic particle hyperthermia—biophysical limitations of a visionary tumour therapy." Journal of Magnetism and Magnetic Materials.
3. Carrey, J., et al. (2011). "Simple models for dynamic hysteresis loop calculations of magnetic single-domain nanoparticles." Journal of Applied Physics.

## Interactive Parameter Adjustment

You can modify the parameters in the cells above and re-run the simulations to see how they affect:
- SAR values
- Temperature evolution
- Treatment efficacy

Try experimenting with:
- Different particle sizes (10-25 nm typical range)
- Various magnetic field strengths (5-30 kA/m)
- Different frequencies (100-500 kHz)
- Nanoparticle concentrations (1-10 kg/m³)